# Business Use Case 4: Product Recommendation System

## Problem Statement
An online store wants to recommend products to users based on their past purchases and interactions.

## Objective
Build a production-ready recommendation system to:
- Recommend relevant products based on user behavior
- Improve customer engagement and sales
- Support personalization at scale
- Enable batch and real-time predictions

## Approach
- **User-Based Collaborative Filtering**: Recommend products liked by similar users
- **Item-Based Collaborative Filtering**: Recommend products similar to user's purchases
- **Content-Based Filtering**: Recommend products from preferred categories/brands
- **Hybrid Approach**: Combine multiple algorithms for better results

## Section 1: Setup and Library Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.sparse import csr_matrix, lil_matrix
from datetime import datetime, timedelta
import json
import os
from typing import List, Dict, Tuple

# Detect environment
is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ
runtime_info = f"Running on: {'Databricks' if is_databricks else 'Local Python'}"
print(f"✓ Setup Complete - {runtime_info}")
print(f"✓ pandas {pd.__version__}, numpy {np.__version__}")

## Section 2: Load and Validate Data

In [ ]:
# Set data path (works for both local and Databricks)
if is_databricks:
    DATA_PATH = "/dbfs/FileStore/product_recommendation_dataset_v2.csv"
else:
    DATA_PATH = "product_recommendation_dataset_v2.csv"

# Load dataset
print(f"📂 Loading data from: {DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH)

print(f"✓ Dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"\nColumn names:")
print(list(df_raw.columns))

# Display data info
print(f"\nData Info:")
print(df_raw.info())
print(f"\nFirst few rows:")
df_raw.head()

In [ ]:
# Schema validation
required_columns = ['user_id', 'product_id', 'interaction_type', 'purchase', 'rating', 'satisfaction_score']
missing_cols = [col for col in required_columns if col not in df_raw.columns]

if missing_cols:
    print(f"⚠️ Warning: Missing columns - {missing_cols}")
else:
    print(f"✓ All required columns present")

# Basic statistics
print(f"\n📊 Basic Statistics:")
print(f"  Unique users: {df_raw['user_id'].nunique()}")
print(f"  Unique products: {df_raw['product_id'].nunique()}")
print(f"  Unique categories: {df_raw['category'].nunique()}")
print(f"  Interaction types: {df_raw['interaction_type'].unique()}")
print(f"  Purchase rate: {(df_raw['purchase'] == 1).sum() / len(df_raw) * 100:.2f}%")

## Section 3: Data Cleaning and Interaction Signal Creation

In [ ]:
# Make a working copy
df = df_raw.copy()

# Handle missing values in key columns
print("🧹 Data Cleaning...")

# Fill missing ratings and satisfaction with median
df['rating'].fillna(df['rating'].median(), inplace=True)
df['satisfaction_score'].fillna(df['satisfaction_score'].median(), inplace=True)
df['final_price_inr'].fillna(df['listed_price_inr'], inplace=True)

# Fill missing purchase values with 0
df['purchase'].fillna(0, inplace=True)
df['review_given'].fillna(0, inplace=True)

# Convert date column if needed
if 'interaction_date' in df.columns:
    df['interaction_date'] = pd.to_datetime(df['interaction_date'])

print(f"✓ Cleaned {len(df_raw) - len(df)} invalid rows")
print(f"✓ Final dataset: {df.shape[0]} rows")

In [ ]:
# Create weighted interaction score
print("\n📊 Creating interaction signals...")

# Base scoring by interaction type
interaction_weights = {
    'purchase': 5.0,
    'review_given': 3.0,
    'add_to_cart': 2.0,
    'wishlist': 1.0,
    'view': 0.5
}

df['interaction_score'] = df['interaction_type'].map(interaction_weights).fillna(0.5)

# Additional boost for reviews
df['interaction_score'] = df['interaction_score'] + (df['review_given'] * 1.0)

# Normalize to 1-5 scale
df['interaction_score'] = df['interaction_score'].clip(upper=5.0)

# Add engagement metrics
df['engagement_score'] = (
    (df['session_duration_sec'] / df['session_duration_sec'].max()) * 0.2 +
    (df['pages_visited'] / df['pages_visited'].max()) * 0.2 +
    (df['rating'] / 5.0) * 0.3 +
    (df['satisfaction_score'] / 10.0) * 0.3
)

# Final composite score
df['final_score'] = (df['interaction_score'] * 0.6 + df['engagement_score'] * 0.4).round(3)

print(f"✓ Interaction signals created")
print(f"\nScore distribution:")
print(df['final_score'].describe())

## Section 4: Build User-Product Interaction Dataset

In [ ]:
# Aggregate interactions by user-product pair
print("🔨 Building interaction aggregates...")

# Create user-product interaction table
interactions = df.groupby(['user_id', 'product_id']).agg({
    'final_score': 'mean',
    'interaction_score': 'mean',
    'engagement_score': 'mean',
    'product_name': 'first',
    'category': 'first',
    'brand': 'first',
    'listed_price_inr': 'first',
    'purchase': 'max',  # 1 if any purchase, 0 otherwise
    'rating': 'mean',
    'satisfaction_score': 'mean',
    'user_segment': 'first',
    'membership_tier': 'first'
}).reset_index()

interactions.columns = ['user_id', 'product_id', 'avg_final_score', 'avg_interaction_score', 
                        'avg_engagement_score', 'product_name', 'category', 'brand', 
                        'price', 'purchased', 'avg_rating', 'avg_satisfaction', 
                        'user_segment', 'membership_tier']

print(f"✓ Created {len(interactions)} user-product interactions")
print(f"\nInteraction statistics:")
print(interactions[['avg_final_score', 'purchased', 'avg_rating']].describe())
print(f"\nSample interactions:")
interactions.head(10)

## Section 5: Create User-Product Matrices

In [ ]:
# Pivot to dense matrix (for smaller datasets)
print("📊 Creating user-product matrices...")

# Dense matrix
user_product_matrix = interactions.pivot_table(
    index='user_id',
    columns='product_id',
    values='avg_final_score',
    fill_value=0
)

print(f"✓ Dense matrix shape: {user_product_matrix.shape}")
print(f"  Sparsity: {(user_product_matrix == 0).sum().sum() / (user_product_matrix.shape[0] * user_product_matrix.shape[1]) * 100:.2f}%")

# Create sparse matrix for efficient computation
sparse_matrix = csr_matrix(user_product_matrix.values)
print(f"✓ Sparse matrix size: {sparse_matrix.data.nbytes / 1024:.2f} KB")

# Store mapping dictionaries
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_product_matrix.index)}
idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}

product_to_idx = {product_id: idx for idx, product_id in enumerate(user_product_matrix.columns)}
idx_to_product = {idx: product_id for product_id, idx in product_to_idx.items()}

print(f"✓ Created mapping dictionaries")
print(f"  Users: {len(user_to_idx)}")
print(f"  Products: {len(product_to_idx)}")

## Section 6: Train-Test Split for Evaluation

In [ ]:
# Create train-test split (leave-one-out by user)
print("🔄 Creating train-test split...")

# For evaluation: hold out interactions from 20% of users
all_users = interactions['user_id'].unique()
test_user_count = max(1, int(len(all_users) * 0.2))

np.random.seed(42)
test_users = np.random.choice(all_users, size=test_user_count, replace=False)
train_users = np.setdiff1d(all_users, test_users)

train_interactions = interactions[interactions['user_id'].isin(train_users)].copy()
test_interactions = interactions[interactions['user_id'].isin(test_users)].copy()

# Create training matrix
train_matrix = train_interactions.pivot_table(
    index='user_id',
    columns='product_id',
    values='avg_final_score',
    fill_value=0
)

print(f"✓ Train-test split created:")
print(f"  Training users: {len(train_users)} ({len(train_interactions)} interactions)")
print(f"  Test users: {len(test_users)} ({len(test_interactions)} interactions)")
print(f"  Train matrix shape: {train_matrix.shape}")

## Section 7: User-Based Collaborative Filtering

In [ ]:
# Compute user-user similarity matrix
print("🤖 Computing user-user similarity...")

# Using cosine similarity on the training matrix
user_similarity = cosine_similarity(train_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=train_matrix.index,
    columns=train_matrix.index
)

print(f"✓ User similarity matrix: {user_similarity.shape}")
print(f"\nSample similarity scores (first user vs others):")
print(user_similarity_df.iloc[0].sort_values(ascending=False).head(10))

In [ ]:
# Function to get user-based recommendations
def get_user_based_recommendations(user_id, n_recommendations=5, n_similar_users=10):
    """
    Generate recommendations for a user based on similar users' preferences.
    """
    if user_id not in user_similarity_df.index:
        # New user: return popular products
        return interactions.nlargest(n_recommendations, 'avg_final_score')[['product_id', 'product_name', 'category', 'avg_final_score']].to_dict('records')
    
    # Get similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:n_similar_users+1].index.tolist()
    
    # Get products rated by similar users
    user_interactions = interactions[interactions['user_id'] == user_id]['product_id'].tolist()
    
    recommendations = {}
    for sim_user in similar_users:
        sim_user_products = interactions[interactions['user_id'] == sim_user]
        for _, row in sim_user_products.iterrows():
            if row['product_id'] not in user_interactions:
                sim_score = user_similarity_df.loc[user_id, sim_user]
                if row['product_id'] not in recommendations:
                    recommendations[row['product_id']] = {
                        'score': 0,
                        'product_name': row['product_name'],
                        'category': row['category'],
                        'brand': row['brand'],
                        'price': row['price'],
                        'rating': row['avg_rating']
                    }
                recommendations[row['product_id']]['score'] += sim_score * row['avg_final_score']
    
    # Sort by score and return top N
    sorted_recs = sorted(recommendations.items(), key=lambda x: x[1]['score'], reverse=True)
    result = []
    for product_id, rec_data in sorted_recs[:n_recommendations]:
        rec_data['product_id'] = product_id
        result.append(rec_data)
    return result

print("✓ User-based recommendation function created")

## Section 8: Item-Based Collaborative Filtering

In [ ]:
# Compute item-item similarity matrix
print("🎯 Computing product-product similarity...")
product_similarity = cosine_similarity(train_matrix.T)
product_similarity_df = pd.DataFrame(
    product_similarity,
    index=train_matrix.columns,
    columns=train_matrix.columns
)

print(f"✓ Product similarity matrix: {product_similarity.shape}")

def get_item_based_recommendations(user_id, n_recommendations=5):
    """
    Generate recommendations based on products similar to user's purchases.
    """
    user_interactions = interactions[interactions['user_id'] == user_id]['product_id'].tolist()
    
    if not user_interactions:
        # New user: return popular products
        return interactions.nlargest(n_recommendations, 'avg_final_score')[['product_id', 'product_name', 'category', 'avg_final_score']].to_dict('records')
    
    recommendations = {}
    for product in user_interactions:
        if product in product_similarity_df.index:
            similar_products = product_similarity_df[product].sort_values(ascending=False)[1:21].index.tolist()
            for sim_product in similar_products:
                if sim_product not in user_interactions:
                    sim_score = product_similarity_df.loc[product, sim_product]
                    if sim_product not in recommendations:
                        prod_info = interactions[interactions['product_id'] == sim_product].iloc[0]
                        recommendations[sim_product] = {
                            'score': 0,
                            'product_name': prod_info['product_name'],
                            'category': prod_info['category'],
                            'brand': prod_info['brand'],
                            'price': prod_info['price'],
                            'rating': prod_info['avg_rating']
                        }
                    recommendations[sim_product]['score'] += sim_score
    
    sorted_recs = sorted(recommendations.items(), key=lambda x: x[1]['score'], reverse=True)
    result = []
    for product_id, rec_data in sorted_recs[:n_recommendations]:
        rec_data['product_id'] = product_id
        result.append(rec_data)
    return result

print("✓ Item-based recommendation function created")

## Section 9: Content-Based Filtering

In [ ]:
# Content-based filtering using categories and brands
print("📦 Setting up content-based filtering...")

def get_content_based_recommendations(user_id, n_recommendations=5):
    """
    Generate recommendations based on product categories and brands user likes.
    """
    user_products = interactions[interactions['user_id'] == user_id]
    
    if len(user_products) == 0:
        # New user: return popular products by overall rating
        return interactions.nlargest(n_recommendations, 'avg_rating')[['product_id', 'product_name', 'category', 'avg_rating']].to_dict('records')
    
    # Identify user preferences
    preferred_categories = user_products['category'].value_counts().head(3).index.tolist()
    preferred_brands = user_products['brand'].value_counts().head(3).index.tolist()
    
    # Get unseen products in preferred categories/brands
    seen_products = set(user_products['product_id'].tolist())
    candidates = interactions[
        ((interactions['category'].isin(preferred_categories)) |
         (interactions['brand'].isin(preferred_brands))) &
        (~interactions['product_id'].isin(seen_products))
    ]
    
    if len(candidates) == 0:
        # Fallback to popular products
        return interactions[~interactions['product_id'].isin(seen_products)].nlargest(n_recommendations, 'avg_rating')[['product_id', 'product_name', 'category', 'avg_rating']].to_dict('records')
    
    # Score candidates
    candidates['content_score'] = (
        (candidates['category'].isin(preferred_categories).astype(float) * 0.4) +
        (candidates['brand'].isin(preferred_brands).astype(float) * 0.3) +
        ((candidates['avg_rating'] / 5.0) * 0.3)
    )
    
    top_recs = candidates.nlargest(n_recommendations, 'content_score')[[
        'product_id', 'product_name', 'category', 'brand', 'price', 'avg_rating'
    ]].to_dict('records')
    
    return top_recs

print("✓ Content-based recommendation function created")

## Section 10: Hybrid Recommendations

In [ ]:
def get_hybrid_recommendations(user_id, n_recommendations=5):
    """
    Generate recommendations using hybrid approach (combining multiple algorithms).
    """
    # Get recommendations from each method
    user_cf_recs = get_user_based_recommendations(user_id, n_recommendations=10)
    item_cf_recs = get_item_based_recommendations(user_id, n_recommendations=10)
    content_recs = get_content_based_recommendations(user_id, n_recommendations=10)
    
    # Combine scores
    combined_scores = {}
    
    # Weight: user-CF (30%), item-CF (30%), content (40%)
    for rec in user_cf_recs:
        product_id = rec['product_id']
        combined_scores[product_id] = combined_scores.get(product_id, {}).copy()
        combined_scores[product_id]['user_cf_score'] = rec.get('score', 0) * 0.3
        combined_scores[product_id].update({k: v for k, v in rec.items() if k != 'score'})
    
    for rec in item_cf_recs:
        product_id = rec['product_id']
        if product_id not in combined_scores:
            combined_scores[product_id] = {}
        combined_scores[product_id]['item_cf_score'] = rec.get('score', 0) * 0.3
        combined_scores[product_id].update({k: v for k, v in rec.items() if k != 'score'})
    
    for rec in content_recs:
        product_id = rec['product_id']
        if product_id not in combined_scores:
            combined_scores[product_id] = {}
        # Use avg_rating or 0 if not present
        content_score = rec.get('avg_rating', rec.get('content_score', 0)) / 5.0 * 0.4
        combined_scores[product_id]['content_score'] = content_score
        combined_scores[product_id].update({k: v for k, v in rec.items() if k not in ['score', 'content_score']})
    
    # Calculate final scores
    for product_id in combined_scores:
        scores = [
            combined_scores[product_id].get('user_cf_score', 0),
            combined_scores[product_id].get('item_cf_score', 0),
            combined_scores[product_id].get('content_score', 0)
        ]
        combined_scores[product_id]['final_score'] = sum(scores)
    
    # Sort and return top N
    sorted_recs = sorted(combined_scores.items(), key=lambda x: x[1].get('final_score', 0), reverse=True)
    result = []
    for product_id, rec_data in sorted_recs[:n_recommendations]:
        rec_data['product_id'] = product_id
        result.append(rec_data)
    return result

print("✓ Hybrid recommendation function created")

## Section 11: Example Recommendations

In [ ]:
# Get a sample user
sample_user = interactions['user_id'].iloc[0]
print(f"\n🎯 Generating recommendations for user: {sample_user}")

# User details
user_detail = df_raw[df_raw['user_id'] == sample_user].iloc[0]
print(f"\n👤 User Profile:")
print(f"  Segment: {user_detail['user_segment']}")
print(f"  Age: {user_detail['age_group']}")
print(f"  Membership: {user_detail['membership_tier']}")
print(f"  Location: {user_detail['location']}")

# Get recommendations using different methods
print(f"\n📌 User-Based Collaborative Filtering Recommendations:")
user_cf_recs = get_user_based_recommendations(sample_user, n_recommendations=3)
for i, rec in enumerate(user_cf_recs, 1):
    print(f"  {i}. {rec['product_name']} ({rec['category']}) - Score: {rec['score']:.3f}")

print(f"\n🔗 Item-Based Collaborative Filtering Recommendations:")
item_cf_recs = get_item_based_recommendations(sample_user, n_recommendations=3)
for i, rec in enumerate(item_cf_recs, 1):
    print(f"  {i}. {rec['product_name']} ({rec['category']}) - Score: {rec['score']:.3f}")

print(f"\n📦 Content-Based Recommendations:")
content_recs = get_content_based_recommendations(sample_user, n_recommendations=3)
for i, rec in enumerate(content_recs, 1):
    print(f"  {i}. {rec['product_name']} ({rec['category']}) - Rating: {rec.get('avg_rating', rec.get('rating', 0)):.1f}")

print(f"\n⭐ Hybrid Recommendations (Final):")
hybrid_recs = get_hybrid_recommendations(sample_user, n_recommendations=5)
for i, rec in enumerate(hybrid_recs, 1):
    print(f"  {i}. {rec['product_name']} ({rec['category']}) - Final Score: {rec.get('final_score', 0):.3f}")

## Section 12: Batch Recommendations for All Users

In [ ]:
print("\n🔄 Generating batch recommendations for all users...")

all_recommendations = []
all_users = interactions['user_id'].unique()

for idx, user_id in enumerate(all_users):
    if (idx + 1) % 100 == 0:
        print(f"  Progress: {idx + 1}/{len(all_users)}")
    
    # Get hybrid recommendations
    recs = get_hybrid_recommendations(user_id, n_recommendations=5)
    
    for rank, rec in enumerate(recs, 1):
        all_recommendations.append({
            'user_id': user_id,
            'product_id': rec['product_id'],
            'product_name': rec['product_name'],
            'category': rec.get('category', ''),
            'brand': rec.get('brand', ''),
            'price': rec.get('price', 0),
            'rating': rec.get('rating', rec.get('avg_rating', 0)),
            'score': rec.get('final_score', 0),
            'rank': rank,
            'timestamp': datetime.now().isoformat()
        })

recommendations_df = pd.DataFrame(all_recommendations)
print(f"\n✓ Generated {len(recommendations_df)} recommendations")
print(f"\nRecommendation statistics:")
print(recommendations_df.groupby('rank').size())
print(f"\nSample recommendations:")
recommendations_df.head(10)

## Section 13: Evaluation Metrics

In [ ]:
# Evaluate on test set
print("\n📊 Evaluating Recommendations...")

def calculate_metrics(test_users_sample, k=5):
    """
    Calculate precision@k and recall@k for recommendations.
    """
    precisions = []
    recalls = []
    ndcgs = []
    
    for user_id in test_users_sample[:50]:
        # Get test interactions (relevant items)
        test_user_products = set(test_interactions[test_interactions['user_id'] == user_id]['product_id'].tolist())
        
        if len(test_user_products) == 0:
            continue
        
        # Get recommendations
        recs = get_hybrid_recommendations(user_id, n_recommendations=k)
        rec_products = [rec['product_id'] for rec in recs]
        
        # Calculate precision@k
        relevant_in_recs = len(set(rec_products) & test_user_products)
        precision = relevant_in_recs / min(k, len(rec_products)) if rec_products else 0
        precisions.append(precision)
        
        # Calculate recall@k
        recall = relevant_in_recs / len(test_user_products) if test_user_products else 0
        recalls.append(recall)
    
    return {
        'precision@k': np.mean(precisions) if precisions else 0,
        'recall@k': np.mean(recalls) if recalls else 0
    }

if len(test_users) > 0:
    metrics = calculate_metrics(test_users, k=5)
    print(f"\n🎯 Evaluation Results (K=5):")
    print(f"  Precision@5: {metrics['precision@k']:.4f}")
    print(f"  Recall@5: {metrics['recall@k']:.4f}")
    print(f"  F1-Score: {2 * (metrics['precision@k'] * metrics['recall@k']) / (metrics['precision@k'] + metrics['recall@k'] + 1e-6):.4f}")
else:
    print("No test users available for evaluation")

## Section 14: Cold-Start Handling

In [ ]:
def get_popular_products(n_recommendations=5, category=None, segment=None):
    """
    Get popular products for cold-start scenarios.
    """
    candidates = interactions.copy()
    
    if category:
        candidates = candidates[candidates['category'] == category]
    
    if len(candidates) == 0:
        candidates = interactions.copy()
    
    # Score by purchases and rating
    candidates['popularity_score'] = (
        (candidates['purchased'].astype(float) * 0.5) +
        ((candidates['avg_rating'] / 5.0) * 0.5)
    )
    
    return candidates.nlargest(n_recommendations, 'popularity_score')[[
        'product_id', 'product_name', 'category', 'brand', 'price', 'avg_rating'
    ]].to_dict('records')

print("✓ Cold-start recommendation functions created")

# Example: New user in Electronics category
print(f"\n❄️ Cold-Start Recommendations (New User - Electronics):")
cold_start_recs = get_popular_products(n_recommendations=5, category='Electronics')
for i, rec in enumerate(cold_start_recs, 1):
    print(f"  {i}. {rec['product_name']} - ⭐ {rec['avg_rating']:.1f}")

## Section 15: Save Recommendations and Summary

In [ ]:
# Save recommendations to CSV
output_path = 'recommendations_output.csv'
recommendations_df.to_csv(output_path, index=False)
print(f"\n💾 Saved recommendations to: {output_path}")

# Generate summary statistics
print(f"\n📈 Recommendation System Summary:")
print(f"\n  Dataset:")
print(f"    Total records: {len(df):,}")
print(f"    Unique users: {df['user_id'].nunique():,}")
print(f"    Unique products: {df['product_id'].nunique():,}")
print(f"    Unique categories: {df['category'].nunique()}")
print(f"    Purchase rate: {(df['purchase'] == 1).sum() / len(df) * 100:.2f}%")

print(f"\n  Model Performance:")
print(f"    Recommendations generated: {len(recommendations_df):,}")
print(f"    Avg recommendations per user: {len(recommendations_df) / df['user_id'].nunique():.1f}")
print(f"    Coverage: {recommendations_df['product_id'].nunique() / df['product_id'].nunique() * 100:.1f}%")

print(f"\n  Top Recommended Products:")
top_products = recommendations_df['product_name'].value_counts().head(5)
for product, count in top_products.items():
    print(f"    • {product}: {count} recommendations")

print(f"\n✅ Recommendation System Complete!")